## The target:Convert AR6 land use data into a suitable format for demeter
### AR6 data only has country landclass data, while Demeter's standard format is basin level.
### 1. Mapping AR6'countries to Demeter'basins (Get 'Metric_id')
### 2. Mapping AR6'countries to Demeter'region (Get 'Region')
### 3. Given that the AR6 land use data only provides land use data in country level, we needed to bring in a weighting factor in order to assign these country level data to the basin level. We chose to use the proportion of the area of each basin in the Demeter projected data table between 1995 and 2015 as a weighting factor. Specifically, we will use this proportion to adjust the AR6 land use data to reflect the actual situation at the watershed level.

In [1]:
import demeter
import pandas as pd

In [279]:
help(demeter)

Help on package demeter:

NAME
    demeter - Demeter:  a land use land cover disaggregation and change detection model.

DESCRIPTION
    Copyright (c) 2017, Battelle Memorial Institute
    
    Open source under license BSD 2-Clause - see LICENSE and DISCLAIMER
    
    @author:  Chris R. Vernon (PNNL); Yannick le Page (niquya@gmail.com)

PACKAGE CONTENTS
    change (package)
    config_reader
    constraints
    demeter_io (package)
    install_supplement
    logger
    model
    ncdf_conversion
    preprocess_data
    process
    reconcile
    staging
    tests (package)
    weight (package)

CLASSES
    builtins.object
        demeter.model.Model
        demeter.preprocess_data.FormatGcamDataFrame
    
    class FormatGcamDataFrame(builtins.object)
     |  FormatGcamDataFrame(df, f_out=None, start_year=2010, through_year=2100, region_name_field='gcam_region_name', region_id_field='gcam_region_id', basin_name_field='glu_name', basin_id_field='basin_id', output_to_csv=False, gcam_land

In [2]:

config_file = '/home/cgl/LUCC_Climate/demeter/config_gcam_reference.ini'

# run all time steps
demeter.run_model(config_file=config_file,
                  write_outputs=True)

2024-05-27 06:17:03,245 - demeter_runtime - INFO - Using `observed_lu_file`:  /home/cgl/LUCC_Climate/demeter/inputs/observed/gcam_reg32_basin235_modis_v6_2010_5arcmin_sqdeg_wgs84_11Jul2019.zip
2024-05-27 06:17:03,247 - demeter_runtime - INFO - Using `run_dir`:  /home/cgl/LUCC_Climate/demeter
2024-05-27 06:17:03,248 - demeter_runtime - INFO - START
2024-05-27 06:17:03,316 - demeter_runtime - INFO - Reading allocation input files...
2024-05-27 06:17:03,380 - demeter_runtime - INFO - PERFORMANCE:  Allocation files processed in 0.06358599662780762 seconds
2024-05-27 06:17:03,383 - demeter_runtime - INFO - Preparing projected land use data...
2024-05-27 06:17:03,384 - demeter_runtime - INFO - Using projected GCAM data from:  /home/cgl/LUCC_Climate/demeter/inputs/projected/gcam_ref_scenario_reg32_basin235_v5p1p3.csv
2024-05-27 06:17:03,456 - demeter_runtime - INFO - Number of regions from projected file:  32
2024-05-27 06:17:03,457 - demeter_runtime - INFO - Number of basins from projected f

### Corresponds to C1, C2, C3 in different scenarios.

In [2]:
AR6_ISO_database = pd.read_csv('/home/cgl/LUCC_Climate/AR6ISO_database/AR6_Scenarios_Database_ISO3_v1.1.csv')

In [3]:
AR6_scenario = pd.read_excel('/home/cgl/LUCC_Climate/AR6ISO_database/AR6_Scenarios_Database_metadata_indicators_v1.1.xlsx',sheet_name='meta_Ch3vetted_withclimate')

In [6]:
AR6_ISO_database.head()

,Model,Scenario,Region,Variable,Unit,1990,1995,2000,2005,2007,...,2070,2075,2080,2085,2090,2095,2100,2110,2130,2150
0,7see Mk5-20 GB,JPC to 0.8 of asymptote,GBR,Carbon Sequestration|CCS|Fossil,Mt CO2/yr,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,7see Mk5-20 GB,JPC to 0.8 of asymptote,GBR,Consumption,billion US$2010/yr,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,7see Mk5-20 GB,JPC to 0.8 of asymptote,GBR,Emissions|CO2|Energy,Mt CO2/yr,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,7see Mk5-20 GB,JPC to 0.8 of asymptote,GBR,Emissions|CO2|Energy|Demand,Mt CO2/yr,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,7see Mk5-20 GB,JPC to 0.8 of asymptote,GBR,Emissions|CO2|Energy|Demand|Commercial,Mt CO2/yr,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
AR6_scenario.head()

,Model,Scenario,Category,Category_name,Category_subset,Subset_Ch4,Category_Vetting_historical,IMP_marker,Literature Reference (if applicable),Policy_category,...,P67 peak warming (FaIRv1.6.2),Median warming in 2100 (FaIRv1.6.2),Median year of peak warming (FaIRv1.6.2),Exceedance Probability 1.5C (FaIRv1.6.2),Exceedance Probability 2.0C (FaIRv1.6.2),Exceedance Probability 3.0C (FaIRv1.6.2),IMP_color_rgb,IMP_color_hex,Category_color_rgb,Category_color_hex
0,AIM/CGE 2.0,SSP1-26,C3,C3: limit warming to 2°C (>67%),C3y_+veGHGs,Limit to 2C (>67%) immediate 2020 action,C3,non-IMP,https://doi.org/10.1016/j.gloenvcha.2016.05.009,P2a,...,1.717124,1.536621,2070,0.599911,0.121591,0.001788,NaN,NaN,"111, 120, 153",6F7899
1,AIM/CGE 2.0,SSP1-34,C5,C5: limit warming to 2.5°C (>50%),C5,NaN,C5,non-IMP,https://doi.org/10.1016/j.gloenvcha.2016.05.009,P2a,...,2.144655,1.962484,2100,0.931158,0.463120,0.022351,NaN,NaN,"140, 167, 208",8CA7D0
2,AIM/CGE 2.0,SSP1-45,C6,C6: limit warming to 3°C (>50%),C6,NaN,C6,non-IMP,https://doi.org/10.1016/j.gloenvcha.2016.05.009,P2a,...,2.629060,2.405440,2100,0.996424,0.836388,0.144837,NaN,NaN,"250, 193, 130",FAC182
3,AIM/CGE 2.0,SSP1-Baseline,C7,C7: limit warming to 4°C (>50%),C7,NaN,C7,non-IMP,https://doi.org/10.1016/j.gloenvcha.2016.05.009,P1a,...,3.270344,3.002760,2100,1.000000,0.990612,0.502012,NaN,NaN,"241, 136, 114",F18872
4,AIM/CGE 2.0,SSP4-26,C3,C3: limit warming to 2°C (>67%),C3y_+veGHGs,Limit to 2C (>67%) immediate 2020 action,C3,non-IMP,https://doi.org/10.1016/j.gloenvcha.2016.05.009,P2a,...,1.678820,1.496835,2070,0.580688,0.082700,0.000447,NaN,NaN,"111, 120, 153",6F7899


In [7]:
AR6_scenario_mapping = AR6_scenario.iloc[:,[1,2]]

In [8]:
AR6_scenario_mapping

,Scenario,Category
0,SSP1-26,C3
1,SSP1-34,C5
2,SSP1-45,C6
3,SSP1-Baseline,C7
4,SSP4-26,C3
...,...,...
1197,CD-LINKS_NPi,C7
1198,CD-LINKS_NPi2020_1000,C1
1199,CD-LINKS_NPi2020_1600,C3
1200,CD-LINKS_NPi2020_400,C1


In [9]:
AR6_ISO_scenario = pd.merge(AR6_ISO_database,AR6_scenario_mapping,on='Scenario')

In [10]:
AR6_ISO_scenario.head()

,Model,Scenario,Region,Variable,Unit,1990,1995,2000,2005,2007,...,2075,2080,2085,2090,2095,2100,2110,2130,2150,Category
0,AIM/CGE 2.2,EN_INDCi2030_1000f,BRA,Agricultural Demand,million t DM/yr,NaN,NaN,NaN,NaN,NaN,...,1985.0627,2027.1252,2076.0882,2124.6306,2168.1395,2205.4125,NaN,NaN,NaN,C3
1,AIM/CGE 2.2,EN_INDCi2030_1000f,BRA,Agricultural Demand,million t DM/yr,NaN,NaN,NaN,NaN,NaN,...,1985.0627,2027.1252,2076.0882,2124.6306,2168.1395,2205.4125,NaN,NaN,NaN,C5
2,AIM/CGE 2.2,EN_INDCi2030_1000f,BRA,Agricultural Demand,million t DM/yr,NaN,NaN,NaN,NaN,NaN,...,1985.0627,2027.1252,2076.0882,2124.6306,2168.1395,2205.4125,NaN,NaN,NaN,C4
3,AIM/CGE 2.2,EN_INDCi2030_1000f,BRA,Agricultural Demand,million t DM/yr,NaN,NaN,NaN,NaN,NaN,...,1985.0627,2027.1252,2076.0882,2124.6306,2168.1395,2205.4125,NaN,NaN,NaN,C4
4,AIM/CGE 2.2,EN_INDCi2030_1000f,BRA,Agricultural Demand,million t DM/yr,NaN,NaN,NaN,NaN,NaN,...,1985.0627,2027.1252,2076.0882,2124.6306,2168.1395,2205.4125,NaN,NaN,NaN,C4


In [11]:
AR6_ISO_scenario = AR6_ISO_scenario.drop(['Model','Scenario','Unit','2110','2130','2150'],axis=1)

In [12]:
AR6_ISO_scenario = AR6_ISO_scenario[AR6_ISO_scenario['Category'].isin(['C1', 'C2', 'C3'])]

In [13]:
AR6_ISO_scenario_group = AR6_ISO_scenario.groupby(['Region','Variable','Category']).mean()
AR6_ISO_scenario_group = AR6_ISO_scenario_group.reset_index()

In [22]:
AR6_ISO_scenario_group

,Region,Variable,Category,1990,1995,2000,2005,2007,2008,2010,...,2055,2060,2065,2070,2075,2080,2085,2090,2095,2100
0,AGO,Capacity Additions|Electricity|Biomass|w/ CCS,C3,NaN,NaN,NaN,0.000000,NaN,NaN,0.000000,...,NaN,0.091391,NaN,0.114313,NaN,0.108199,NaN,0.069936,NaN,0.129196
1,AGO,Capacity Additions|Electricity|Biomass|w/o CCS,C3,NaN,NaN,NaN,0.000000,NaN,NaN,0.000000,...,NaN,0.005389,NaN,0.007545,NaN,0.033132,NaN,0.013892,NaN,0.017659
2,AGO,Capacity Additions|Electricity|Coal|w/ CCS,C3,NaN,NaN,NaN,0.000000,NaN,NaN,0.000000,...,NaN,0.008609,NaN,0.012538,NaN,0.018301,NaN,0.023684,NaN,0.030021
3,AGO,Capacity Additions|Electricity|Coal|w/o CCS,C3,NaN,NaN,NaN,0.000000,NaN,NaN,0.000000,...,NaN,0.009075,NaN,0.012624,NaN,0.015843,NaN,0.019854,NaN,0.024919
4,AGO,Capacity Additions|Electricity|Gas|w/ CCS,C3,NaN,NaN,NaN,0.000000,NaN,NaN,0.000000,...,NaN,0.008910,NaN,0.012823,NaN,0.018837,NaN,0.024488,NaN,0.030374
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50276,ZAF,Yield|Cereal,C3,NaN,NaN,NaN,3.041958,NaN,NaN,3.492645,...,5.959575,8.264616,6.226029,9.161894,6.466307,10.136785,6.692883,11.157201,6.817957,11.879810
50277,ZAF,Yield|Oilcrops,C2,NaN,NaN,NaN,1.624933,NaN,NaN,1.858789,...,NaN,3.553695,NaN,3.954104,NaN,4.351814,NaN,4.792217,NaN,4.829037
50278,ZAF,Yield|Oilcrops,C3,NaN,NaN,NaN,1.627256,NaN,NaN,1.858784,...,NaN,3.468499,NaN,3.858506,NaN,4.266148,NaN,4.679764,NaN,4.830291
50279,ZAF,Yield|Sugarcrops,C2,NaN,NaN,NaN,0.000000,NaN,NaN,0.000000,...,NaN,0.000000,NaN,0.000000,NaN,0.000000,NaN,0.000000,NaN,0.000000


In [14]:
AR6_ISO_scenario_Landcover = AR6_ISO_scenario_group[AR6_ISO_scenario_group['Variable'].str.contains('Land Cover', case=False, na=False)]
AR6_ISO_scenario_Landcover['Variable'] = AR6_ISO_scenario_Landcover['Variable'].str.replace('Land Cover|','',regex=False)

/tmp/ipykernel_103798/84004702.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  AR6_ISO_scenario_Landcover['Variable'] = AR6_ISO_scenario_Landcover['Variable'].str.replace('Land Cover|','',regex=False)


In [18]:
AR6_ISO_scenario_Landcover.head()

,Region,Variable,Category,1990,1995,2000,2005,2007,2008,2010,...,2055,2060,2065,2070,2075,2080,2085,2090,2095,2100
1360,ARG,Land Cover,C1,NaN,NaN,NaN,278.039978,NaN,NaN,277.591484,...,276.246100,277.591496,276.246050,277.591509,276.246050,277.591509,276.246100,277.591496,276.246100,277.591509
1361,ARG,Land Cover,C2,NaN,NaN,NaN,278.039978,NaN,NaN,277.965229,...,276.246100,277.965231,276.246050,277.965233,276.246050,277.965233,276.246100,277.965231,276.246100,277.965233
1362,ARG,Land Cover,C3,NaN,NaN,NaN,278.039978,NaN,NaN,277.979777,...,276.246040,277.979779,276.246060,277.979781,276.246100,277.979781,276.246100,277.979781,276.246100,277.979779
1363,ARG,Built-up Area,C1,NaN,NaN,NaN,0.283604,NaN,NaN,0.341520,...,0.559272,0.382254,0.559272,0.385825,0.559272,0.387229,0.559272,0.387255,0.559272,0.387255
1364,ARG,Built-up Area,C2,NaN,NaN,NaN,0.200123,NaN,NaN,0.222382,...,0.559272,0.285234,0.559272,0.290762,0.559272,0.292936,0.559272,0.292977,0.559272,0.292977


# 1.mapping ISO_country to GCAM_basin(Projected data-Metric_id)

In [19]:
basin_ISO_mapping = pd.read_csv('/home/cgl/LUCC_Climate/AR6ISO_database/basin_to_country_mapping.csv')

In [20]:
basin_ISO_mapping

,GCAM_basin_ID,Basin_long_name,GLU_code,GLU_name,ISO,ISO_NUM,Country_name,desal
0,1,Arctic_Ocean_Islands,GLU001,ArcticIsl,GRL,304,Greenland,0
1,2,Northwest_Territories,GLU002,NWTerr,CAN,124,Canada,0
2,3,Siberia_North_Coast,GLU003,SiberiaN,RUS,643,Russian Federation,0
3,4,Siberia_West_Coast,GLU004,SiberiaW,RUS,643,Russian Federation,0
4,5,Kara_Sea_Coast,GLU005,KaraSea,RUS,643,Russian Federation,0
...,...,...,...,...,...,...,...,...
230,231,Rio_Grande_River_Basin,GLU231,RioGrande,USA,840,United States,1
231,232,New_England_Basin,GLU232,UsaCstNE,USA,840,United States,0
232,233,Mid_Atlantic_Basin,GLU233,UsaCstE,USA,840,United States,0
233,234,Hawaii,GLU234,Hawaii,USA,840,United States,0


In [21]:
basin_ISO_mapping_metric = basin_ISO_mapping[['GCAM_basin_ID','ISO']]

In [24]:
basin_ISO_mapping_metric

,GCAM_basin_ID,ISO
0,1,GRL
1,2,CAN
2,3,RUS
3,4,RUS
4,5,RUS
...,...,...
230,231,USA
231,232,USA
232,233,USA
233,234,USA


In [81]:
AR6_ISO_scenario_metric = pd.merge(AR6_ISO_scenario_Landcover,basin_ISO_mapping_metric,left_on='Region',right_on='ISO')

In [82]:
AR6_ISO_scenario_metric = AR6_ISO_scenario_metric.drop('Region',axis=1)

In [83]:
AR6_ISO_scenario_metric

,Variable,Category,1990,1995,2000,2005,2007,2008,2010,2011,...,2065,2070,2075,2080,2085,2090,2095,2100,GCAM_basin_ID,ISO
0,Land Cover,C1,NaN,NaN,NaN,278.039978,NaN,NaN,277.591484,NaN,...,276.246050,277.591509,276.246050,277.591509,276.246100,277.591496,276.246100,277.591509,203,ARG
1,Land Cover,C1,NaN,NaN,NaN,278.039978,NaN,NaN,277.591484,NaN,...,276.246050,277.591509,276.246050,277.591509,276.246100,277.591496,276.246100,277.591509,205,ARG
2,Land Cover,C1,NaN,NaN,NaN,278.039978,NaN,NaN,277.591484,NaN,...,276.246050,277.591509,276.246050,277.591509,276.246100,277.591496,276.246100,277.591509,209,ARG
3,Land Cover,C1,NaN,NaN,NaN,278.039978,NaN,NaN,277.591484,NaN,...,276.246050,277.591509,276.246050,277.591509,276.246100,277.591496,276.246100,277.591509,210,ARG
4,Land Cover,C1,NaN,NaN,NaN,278.039978,NaN,NaN,277.591484,NaN,...,276.246050,277.591509,276.246050,277.591509,276.246100,277.591496,276.246100,277.591509,212,ARG
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6924,Pasture,C2,NaN,NaN,NaN,85.554464,NaN,NaN,88.399032,NaN,...,87.614032,77.315918,87.413057,76.578450,87.288749,75.917420,87.033399,75.305567,204,ZAF
6925,Pasture,C3,NaN,NaN,NaN,84.300863,NaN,NaN,88.058518,NaN,...,88.962223,78.023000,88.947652,77.387328,88.981321,76.811436,88.954217,76.388436,194,ZAF
6926,Pasture,C3,NaN,NaN,NaN,84.300863,NaN,NaN,88.058518,NaN,...,88.962223,78.023000,88.947652,77.387328,88.981321,76.811436,88.954217,76.388436,198,ZAF
6927,Pasture,C3,NaN,NaN,NaN,84.300863,NaN,NaN,88.058518,NaN,...,88.962223,78.023000,88.947652,77.387328,88.981321,76.811436,88.954217,76.388436,199,ZAF


In [84]:
AR6_ISO_scenario_metric.to_csv('/home/cgl/LUCC_Climate/AR6ISO_database/AR6_ISO_scenario_metric.csv')

# 2.mapping GCAM_region to AR6_country

#### 2.1 mapping ISO_country to GCAM_region(Projected data-Region)

In [85]:
AR6_ISO_scenario_metric_new = pd.read_csv('/home/cgl/LUCC_Climate/AR6ISO_database/AR6_ISO_scenario_metric.csv')

In [86]:
basin_ISO_mapping_country = basin_ISO_mapping[['ISO','Country_name']]

In [87]:
basin_ISO_mapping

,GCAM_basin_ID,Basin_long_name,GLU_code,GLU_name,ISO,ISO_NUM,Country_name,desal
0,1,Arctic_Ocean_Islands,GLU001,ArcticIsl,GRL,304,Greenland,0
1,2,Northwest_Territories,GLU002,NWTerr,CAN,124,Canada,0
2,3,Siberia_North_Coast,GLU003,SiberiaN,RUS,643,Russian Federation,0
3,4,Siberia_West_Coast,GLU004,SiberiaW,RUS,643,Russian Federation,0
4,5,Kara_Sea_Coast,GLU005,KaraSea,RUS,643,Russian Federation,0
...,...,...,...,...,...,...,...,...
230,231,Rio_Grande_River_Basin,GLU231,RioGrande,USA,840,United States,1
231,232,New_England_Basin,GLU232,UsaCstNE,USA,840,United States,0
232,233,Mid_Atlantic_Basin,GLU233,UsaCstE,USA,840,United States,0
233,234,Hawaii,GLU234,Hawaii,USA,840,United States,0


In [88]:
basin_ISO_mapping_country[basin_ISO_mapping_country['ISO']=='ARG']

,ISO,Country_name
202,ARG,Argentina
204,ARG,Argentina
208,ARG,Argentina
209,ARG,Argentina
211,ARG,Argentina
212,ARG,Argentina
213,ARG,Argentina
214,ARG,Argentina


In [89]:
AR6_ISO_scenario_metric_country = pd.merge(AR6_ISO_scenario_metric,basin_ISO_mapping_country,on='ISO')

In [90]:
AR6_ISO_scenario_metric_country.to_csv('/home/cgl/LUCC_Climate/AR6ISO_database/AR6_ISO_scenario_metric_country.csv')

In [91]:
AR6_ISO_scenario_metric_country

,Variable,Category,1990,1995,2000,2005,2007,2008,2010,2011,...,2070,2075,2080,2085,2090,2095,2100,GCAM_basin_ID,ISO,Country_name
0,Land Cover,C1,NaN,NaN,NaN,278.039978,NaN,NaN,277.591484,NaN,...,277.591509,276.246050,277.591509,276.246100,277.591496,276.246100,277.591509,203,ARG,Argentina
1,Land Cover,C1,NaN,NaN,NaN,278.039978,NaN,NaN,277.591484,NaN,...,277.591509,276.246050,277.591509,276.246100,277.591496,276.246100,277.591509,203,ARG,Argentina
2,Land Cover,C1,NaN,NaN,NaN,278.039978,NaN,NaN,277.591484,NaN,...,277.591509,276.246050,277.591509,276.246100,277.591496,276.246100,277.591509,203,ARG,Argentina
3,Land Cover,C1,NaN,NaN,NaN,278.039978,NaN,NaN,277.591484,NaN,...,277.591509,276.246050,277.591509,276.246100,277.591496,276.246100,277.591509,203,ARG,Argentina
4,Land Cover,C1,NaN,NaN,NaN,278.039978,NaN,NaN,277.591484,NaN,...,277.591509,276.246050,277.591509,276.246100,277.591496,276.246100,277.591509,203,ARG,Argentina
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84832,Pasture,C3,NaN,NaN,NaN,84.300863,NaN,NaN,88.058518,NaN,...,78.023000,88.947652,77.387328,88.981321,76.811436,88.954217,76.388436,199,ZAF,South Africa
84833,Pasture,C3,NaN,NaN,NaN,84.300863,NaN,NaN,88.058518,NaN,...,78.023000,88.947652,77.387328,88.981321,76.811436,88.954217,76.388436,204,ZAF,South Africa
84834,Pasture,C3,NaN,NaN,NaN,84.300863,NaN,NaN,88.058518,NaN,...,78.023000,88.947652,77.387328,88.981321,76.811436,88.954217,76.388436,204,ZAF,South Africa
84835,Pasture,C3,NaN,NaN,NaN,84.300863,NaN,NaN,88.058518,NaN,...,78.023000,88.947652,77.387328,88.981321,76.811436,88.954217,76.388436,204,ZAF,South Africa


#### 2.2 mapping GCAM_region to AR6_country  

In [92]:
GCAM_region = pd.read_csv('/home/cgl/LUCC_Climate/AR6ISO_database/GCAM_regino_mapping.csv')

In [93]:
GCAM_region.head()

,Country,GCAM_Region
0,Burundi,Africa_Eastern
1,Comoros,Africa_Eastern
2,Djibouti,Africa_Eastern
3,Eritrea,Africa_Eastern
4,Ethiopia,Africa_Eastern


In [94]:
AR6_ISO_scenario_metric_region= pd.merge(AR6_ISO_scenario_metric_country,GCAM_region,left_on='Country_name',right_on='Country')

In [95]:
AR6_ISO_scenario_metric_region.head()

,Variable,Category,1990,1995,2000,2005,2007,2008,2010,2011,...,2080,2085,2090,2095,2100,GCAM_basin_ID,ISO,Country_name,Country,GCAM_Region
0,Land Cover,C1,NaN,NaN,NaN,278.039978,NaN,NaN,277.591484,NaN,...,277.591509,276.2461,277.591496,276.2461,277.591509,203,ARG,Argentina,Argentina,Argentina
1,Land Cover,C1,NaN,NaN,NaN,278.039978,NaN,NaN,277.591484,NaN,...,277.591509,276.2461,277.591496,276.2461,277.591509,203,ARG,Argentina,Argentina,Argentina
2,Land Cover,C1,NaN,NaN,NaN,278.039978,NaN,NaN,277.591484,NaN,...,277.591509,276.2461,277.591496,276.2461,277.591509,203,ARG,Argentina,Argentina,Argentina
3,Land Cover,C1,NaN,NaN,NaN,278.039978,NaN,NaN,277.591484,NaN,...,277.591509,276.2461,277.591496,276.2461,277.591509,203,ARG,Argentina,Argentina,Argentina
4,Land Cover,C1,NaN,NaN,NaN,278.039978,NaN,NaN,277.591484,NaN,...,277.591509,276.2461,277.591496,276.2461,277.591509,203,ARG,Argentina,Argentina,Argentina


In [96]:
AR6_ISO_scenario_metric_region = AR6_ISO_scenario_metric_region.drop_duplicates()

In [97]:
AR6_ISO_scenario_metric_region.to_csv('/home/cgl/LUCC_Climate/AR6ISO_database/AR6_ISO_scenario_metric_region.csv')

In [115]:
AR6_ISO_scenario_metric_region['Variable'].unique()

array(['Land Cover', 'Built-up Area', 'Cropland', 'Cropland|Cereals',
       'Cropland|Energy Crops', 'Cropland|Irrigated', 'Cropland|Rainfed',
       'Forest', 'Forest|Forestry|Harvested Area', 'Forest|Managed',
       'Forest|Natural Forest', 'Other Arable Land', 'Other Land',
       'Pasture', 'Cropland|Double-cropped',
       'Cropland|Energy Crops|Irrigated',
       'Forest|Afforestation and Reforestation',
       'Cropland|Energy Crops|2nd generation', 'Cropland|Oilcrops',
       'Cropland|Sugarcrops', 'Forest|Secondary', 'Forest|Share'],
      dtype=object)

#### 2.3 assign value to GCAM basin (update 10.05)

In [286]:
Demter_projected_ref = pd.read_csv('/home/cgl/LUCC_Climate/demeter/inputs/projected/gcam_ref_scenario_reg32_basin235_v5p1p3.csv')

In [287]:
Demter_projected_ref.head()

,region,landclass,metric_id,1975,1990,2005,2010,2015,2020,2025,...,2055,2060,2065,2070,2075,2080,2085,2090,2095,2100
0,Africa_Eastern,biomass,171,0.0,0.0,0.0,0.0,0.0,1.108500,2.736277,...,10.463834,9.766495,8.978135,8.173274,7.258403,6.480860,5.707765,5.043665,4.587417,4.163356
1,Africa_Eastern,biomass,100,0.0,0.0,0.0,0.0,0.0,0.326966,0.906865,...,3.570573,3.046323,3.021163,2.987825,2.317451,2.217917,1.946624,1.696542,1.528901,1.379762
2,Africa_Eastern,biomass,144,0.0,0.0,0.0,0.0,0.0,0.002230,0.005511,...,0.017220,0.015412,0.015620,0.013098,0.010365,0.007606,0.006393,0.005126,0.004044,0.003618
3,Africa_Eastern,biomass,183,0.0,0.0,0.0,0.0,0.0,0.379626,1.071106,...,5.154627,5.173128,5.089419,4.983678,4.745812,4.578958,4.353882,4.104182,3.922491,3.740382
4,Africa_Eastern,biomass,151,0.0,0.0,0.0,0.0,0.0,2.042369,4.745245,...,13.910337,12.993538,12.152471,11.340450,10.303586,9.365466,8.509158,7.638869,6.891158,6.302276


##### 2.3.1 Using historical data from Demeter as weights for BASIN assignments

##### crop has to add all of crop type in Demeter projected ref table

In [288]:
year_crop = ['1975', '1990', '2005', '2010', '2015', '2020', '2025', '2030', '2035', '2040', '2045', '2050', '2055', '2060', '2065', '2070', '2075', '2080', '2085', '2090', '2095', '2100']


crop_type_Demeter = Demter_projected_ref[Demter_projected_ref['landclass'].isin(['biomass', 'Corn','FiberCrop','FodderGrass','FodderHerb','MiscCrop',
                                                                                         'OilCrop','OtherGrain','PalmFruit','Rice','RootTuber','SugarCrop','Wheat'])]


crop_Demeter = crop_type_Demeter.groupby(['region', 'metric_id'])[year_crop].sum().reset_index()


crop_Demeter['landclass'] = 'CropLand'


Demter_projected_ref = pd.concat([Demter_projected_ref, crop_Demeter], ignore_index=True)


In [299]:
year_columns = [col for col in Demter_projected_ref.columns if col not in ['region', 'landclass', 'metric_id'] and 1975 <= int(col) <= 2015]
columns_of_interest = ['region', 'landclass', 'metric_id'] + year_columns

Demter_projected_ref_sel = Demter_projected_ref[columns_of_interest]

Demter_projected_ref_sel['sum'] = Demter_projected_ref_sel.iloc[:,3:8].sum(axis=1)

Demter_projected_ref_sel.to_csv('/home/cgl/LUCC_Climate/AR6ISO_database/projected_test/Demter_projected_sum.csv')

landclass_totals = Demter_projected_ref_sel.groupby(['region','landclass'])['sum'].sum().reset_index(name='total_sum')
Demter_projected_ref_sel = Demter_projected_ref_sel.merge(landclass_totals, on=['region','landclass'])
Demter_projected_ref_sel.to_csv('/home/cgl/LUCC_Climate/AR6ISO_database/projected_test/Demter_projected_totalsum.csv')
# 计算占比
Demter_projected_ref_sel['weight'] =Demter_projected_ref_sel['sum'] / Demter_projected_ref_sel['total_sum']

/tmp/ipykernel_103798/393350423.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Demter_projected_ref_sel['sum'] = Demter_projected_ref_sel.iloc[:,3:8].sum(axis=1)


In [300]:
Demter_projected_ref_sel

,region,landclass,metric_id,1975,1990,2005,2010,2015,sum,total_sum,weight
0,Africa_Eastern,biomass,171,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
1,Africa_Eastern,biomass,100,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
2,Africa_Eastern,biomass,144,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
3,Africa_Eastern,biomass,183,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
4,Africa_Eastern,biomass,151,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
...,...,...,...,...,...,...,...,...,...,...,...
8663,USA,CropLand,230,9.718212,7.668778,8.251438,7.836923,8.116580,41.591931,5936.711835,0.007006
8664,USA,CropLand,231,5.485849,4.500100,3.873143,3.205875,3.517887,20.582855,5936.711835,0.003467
8665,USA,CropLand,232,3.623327,2.665630,2.503857,2.215672,2.327197,13.335683,5936.711835,0.002246
8666,USA,CropLand,233,36.131180,30.656827,31.048529,29.405945,30.256749,157.499230,5936.711835,0.026530


In [301]:
Demter_projected_ref_sel.to_csv('/home/cgl/LUCC_Climate/AR6ISO_database/Demter_projected_ref_sel.csv')

In [302]:
Demter_projected_ref_sel['landclass'].unique()

array(['biomass', 'Corn', 'FiberCrop', 'FodderGrass', 'FodderHerb',
       'Forest', 'Grassland', 'MiscCrop', 'OilCrop', 'OtherArableLand',
       'OtherGrain', 'PalmFruit', 'Pasture', 'ProtectedGrassland',
       'ProtectedShrubland', 'ProtectedUnmanagedForest',
       'ProtectedUnmanagedPasture', 'Rice', 'RockIceDesert', 'RootTuber',
       'Shrubland', 'SugarCrop', 'UnmanagedForest', 'UnmanagedPasture',
       'UrbanLand', 'Wheat', 'Tundra', 'CropLand'], dtype=object)

In [303]:
AR6_to_drop = ['Cropland|Cereals','Cropland|Energy Crops','Cropland|Irrigated','Cropland|Rainfed','Cropland|Double-cropped',
               'Cropland|Energy Crops|Irrigated','Cropland|Energy Crops|2nd generation','Cropland|Oilcrops','Cropland|Sugarcrops',
               'Forest|Forestry|Harvested Area','Forest|Managed','Forest|Natural Forest','Forest|Afforestation and Reforestation','Forest|Secondary','Forest|Share']

In [304]:
AR6_ISO_scenario_metric_region = AR6_ISO_scenario_metric_region[~AR6_ISO_scenario_metric_region['Variable'].isin(AR6_to_drop)]

##### 2.3.2 assign basin value to AR6  

##### we set Demeter's Urbanland →AR6 Built-up Area，RockIceDeser→Other Land,OtherArableLand→Other Arable Land

In [305]:
Demter_projected_ref_sel['landclass'] = Demter_projected_ref_sel['landclass'].replace({
    'UrbanLand': 'Built-up Area',
    'RockIceDesert': 'Other Land',
    'CropLand': 'Cropland',
    'OtherArableLand': 'Other Arable Land'
})

In [306]:
Demter_projected_ref_sel['landclass'].unique()

array(['biomass', 'Corn', 'FiberCrop', 'FodderGrass', 'FodderHerb',
       'Forest', 'Grassland', 'MiscCrop', 'OilCrop', 'Other Arable Land',
       'OtherGrain', 'PalmFruit', 'Pasture', 'ProtectedGrassland',
       'ProtectedShrubland', 'ProtectedUnmanagedForest',
       'ProtectedUnmanagedPasture', 'Rice', 'Other Land', 'RootTuber',
       'Shrubland', 'SugarCrop', 'UnmanagedForest', 'UnmanagedPasture',
       'Built-up Area', 'Wheat', 'Tundra', 'Cropland'], dtype=object)

In [307]:
AR6_ISO_scenario_metric_region['Variable'].unique()

array(['Land Cover', 'Built-up Area', 'Cropland', 'Forest',
       'Other Arable Land', 'Other Land', 'Pasture'], dtype=object)

In [308]:
AR6_allocation = pd.merge(Demter_projected_ref_sel,AR6_ISO_scenario_metric_region,left_on=['metric_id','landclass'],right_on=['GCAM_basin_ID','Variable'])

In [309]:
AR6_allocation

,region,landclass,metric_id,1975,1990_x,2005_x,2010_x,2015_x,sum,total_sum,...,2080,2085,2090,2095,2100,GCAM_basin_ID,ISO,Country_name,Country,GCAM_Region
0,Africa_Northern,Other Arable Land,75,0.123657,0.150823,0.116395,0.128595,0.121928,0.641398,479.045613,...,10.848092,29.493027,10.835664,29.525931,10.920964,75,TUR,Turkey,Turkey,Europe_Non_EU
1,Africa_Northern,Other Arable Land,75,0.123657,0.150823,0.116395,0.128595,0.121928,0.641398,479.045613,...,6.058200,28.161198,6.041150,28.172115,6.163521,75,TUR,Turkey,Turkey,Europe_Non_EU
2,Africa_Northern,Other Arable Land,75,0.123657,0.150823,0.116395,0.128595,0.121928,0.641398,479.045613,...,7.204416,27.234264,7.066924,27.784820,7.166620,75,TUR,Turkey,Turkey,Europe_Non_EU
3,Africa_Northern,Other Land,75,35.648900,35.577400,35.681300,35.656200,35.656200,178.220000,23481.303340,...,15.018280,0.000000,14.563353,0.000000,14.289545,75,TUR,Turkey,Turkey,Europe_Non_EU
4,Africa_Northern,Other Land,75,35.648900,35.577400,35.681300,35.656200,35.656200,178.220000,23481.303340,...,17.464851,0.000000,16.776787,0.000000,16.258620,75,TUR,Turkey,Turkey,Europe_Non_EU
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2680,USA,Cropland,233,36.131180,30.656827,31.048529,29.405945,30.256749,157.499230,5936.711835,...,217.498401,213.729789,219.922490,215.785013,223.361655,233,USA,United States,United States,USA
2681,USA,Cropland,233,36.131180,30.656827,31.048529,29.405945,30.256749,157.499230,5936.711835,...,208.746769,213.757811,211.464353,217.980899,213.290652,233,USA,United States,United States,USA
2682,USA,Cropland,234,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5936.711835,...,210.572394,189.198551,211.052332,191.841473,211.813456,234,USA,United States,United States,USA
2683,USA,Cropland,234,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5936.711835,...,217.498401,213.729789,219.922490,215.785013,223.361655,234,USA,United States,United States,USA


In [310]:
AR6_allocation = AR6_allocation.drop(['sum','total_sum','Variable','GCAM_basin_ID','ISO','Country_name',
                                      '1990_y','1995','2000','2005_y','2007','2008','2010_y','2011','2012','2013','2014','2015_y','2018','2021','2022','2023','2024',
                                      'Country','GCAM_Region'],axis=1)

In [311]:
AR6_allocation.columns[-17]

'2020'

In [312]:
AR6_allocation.iloc[:,-17:] = AR6_allocation.iloc[:,-17:] .multiply(AR6_allocation['weight'], axis=0)

##### 2.3.3 factor to multipl the AR data (match the Demeter)

In [316]:
AR6_allocation.iloc[:, -10:] = AR6_allocation.iloc[:, -10:] * 10 # factor is 10
AR6_allocation

,region,landclass,metric_id,1975,1990_x,2005_x,2010_x,2015_x,weight,Category,...,2055,2060,2065,2070,2075,2080,2085,2090,2095,2100
0,Africa_Northern,Other Arable Land,75,0.123657,0.150823,0.116395,0.128595,0.121928,0.001339,C1,...,0.436969,0.157834,0.419161,0.150453,0.398218,0.145246,0.394884,0.145080,0.395325,0.146222
1,Africa_Northern,Other Arable Land,75,0.123657,0.150823,0.116395,0.128595,0.121928,0.001339,C2,...,0.424411,0.087416,0.404947,0.081408,0.363378,0.081114,0.377053,0.080885,0.377199,0.082524
2,Africa_Northern,Other Arable Land,75,0.123657,0.150823,0.116395,0.128595,0.121928,0.001339,C3,...,0.433885,0.100939,0.406625,0.096064,0.362512,0.096461,0.364642,0.094620,0.372013,0.095954
3,Africa_Northern,Other Land,75,35.648900,35.577400,35.681300,35.656200,35.656200,0.007590,C1,...,0.000000,1.206293,0.000000,1.180592,0.000000,1.139868,0.000000,1.105339,0.000000,1.084558
4,Africa_Northern,Other Land,75,35.648900,35.577400,35.681300,35.656200,35.656200,0.007590,C2,...,0.000000,1.436385,0.000000,1.386684,0.000000,1.325559,0.000000,1.273336,0.000000,1.234008
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2680,USA,Cropland,233,36.131180,30.656827,31.048529,29.405945,30.256749,0.026530,C2,...,56.864325,55.811099,54.020120,56.857522,56.273225,57.701690,56.701888,58.344794,57.247133,59.257194
2681,USA,Cropland,233,36.131180,30.656827,31.048529,29.405945,30.256749,0.026530,C3,...,54.315204,53.958603,53.470952,54.692620,55.090629,55.379908,56.709323,56.100875,57.829695,56.585387
2682,USA,Cropland,234,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,C1,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2683,USA,Cropland,234,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,C2,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [314]:
AR6_allocation.to_csv('/home/cgl/LUCC_Climate/AR6ISO_database/projected_test/AR6_allocation.csv')